## Human-in-the-Loop Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Some agent actions are too risky, costly, or irreversible to let the agent take unsupervised — sending an email, deleting a record, executing a trade. Human-in-the-Loop middleware pauses the agent before such actions and waits for a human to approve, edit, or reject them before continuing.

- **Interrupt** — pauses execution right before a designated tool call is made, instead of letting it run automatically.
- **Review** — surfaces the pending tool call (name + arguments) to a human for inspection.
- **Decide** — the human can approve as-is, edit the arguments, or reject the call outright.
- **Resume** — once a decision is made, the agent continues the run using that decision as the outcome of the tool call.

In [1]:
import os
from dotenv import load_dotenv

# Change working directory to the parent folder (01_Langchain)
os.chdir(os.path.abspath(".."))
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage


# Define a tool call function
def read_email_tool(email_id:str) ->str:
    """Mock function to read an email by its ID"""
    return f"Email content for ID {email_id}"

def send_email_tool(recipient:str, subject:str, body:str)->str:
    """Mock funciton to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [12]:
agent = create_agent(
    model="gpt-5",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{"allowed_decisions":["approve","edit","reject"]},
                "read_email_tool":False
            }
        )
    ]
)

In [13]:
config = {"configurable":{"thread_id":"test-approve"}}

# Request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you'")]},
    config=config
)

In [14]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you'", additional_kwargs={}, response_metadata={}, id='272df502-ba0f-4542-8e5c-7b73a455522a'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 164, 'prompt_tokens': 179, 'total_tokens': 343, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQsJmDLSSQGetonFuqfZxWkNK6Nre', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c8ba-1407-71f1-adf0-533e4c2e41b7-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@t

In [15]:
## aprpove email 
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )
    print(f"Result:{result['messages'][-1].content}")

Paused! Approving...
Result:Your email has been sent to john@test.com with the subject "Hello" and body "How are you."


In [16]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you'", additional_kwargs={}, response_metadata={}, id='272df502-ba0f-4542-8e5c-7b73a455522a'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 164, 'prompt_tokens': 179, 'total_tokens': 343, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQsJmDLSSQGetonFuqfZxWkNK6Nre', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c8ba-1407-71f1-adf0-533e4c2e41b7-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@t

In [ ]:
## Reject 

## aprpove email 
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )
    print(f"Result:{result['messages'][-1].content}")

In [19]:

# Define a tool call function
def read_email_tool(email_id:str) ->str:
    """Mock function to read an email by its ID"""
    return f"Email content for ID {email_id}"

def send_email_tool(recipient:str, subject:str, body:str)->str:
    """Mock funciton to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agent = create_agent(
    model="gpt-5",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{"allowed_decisions":["approve","edit","reject"]},
                "read_email_tool":False
            }
        )
    ]
)




In [20]:
config = {"configurable":{"thread_id":"test-reject"}}

# Request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you'")]},
    config=config
)

In [21]:
## Reject 

## aprpove email 
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )
    print(f"Result:{result['messages'][-1].content}")

Paused! Approving...
Result:I couldn’t send the email because the send action was canceled. Do you want me to try again?

Current details:
- To: john@test.com
- Subject: Hello
- Body: How are you

Reply “Send it” to proceed, or tell me what to change.


In [22]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you'", additional_kwargs={}, response_metadata={}, id='3a18b03c-bc10-412a-9533-d04934e54ce3'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 164, 'prompt_tokens': 179, 'total_tokens': 343, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQsNSz7e1M3QaCQcGFxkQ4P0zqDmO', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c8bd-9097-7393-ad14-d68ead43cdb5-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@t

In [23]:
## Reject


# Define a tool call function
def read_email_tool(email_id:str) ->str:
    """Mock function to read an email by its ID"""
    return f"Email content for ID {email_id}"

def send_email_tool(recipient:str, subject:str, body:str)->str:
    """Mock funciton to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agent = create_agent(
    model="gpt-5",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{"allowed_decisions":["approve","edit","reject"]},
                "read_email_tool":False
            }
        )
    ]
)




In [24]:
config = {"configurable":{"thread_id":"test-edit"}}

# Request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to wrong@test.com with subject 'Hello' and body 'How are you'")]},
    config=config
)

In [25]:
## edit and approve
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! Editing...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool",
                            "args":{
                                "recipient":"correct@email.com",
                                "subject":"Corrected Subject",
                                "body":"The was edited by human before sending"
                                }
                            }
                    }
                ]
            }
        ),
        config=config
    )
    print(f"Result:{result['messages'][-1].content}")

Paused! Editing...
Result:I attempted to send your email, but an authorized reviewer modified it before sending. It was sent to correct@email.com with subject “Corrected Subject” and body “The was edited by human before sending.”

Would you like me to send your original message to wrong@test.com with subject “Hello” and body “How are you” instead?


In [26]:
result

{'messages': [HumanMessage(content="Send email to wrong@test.com with subject 'Hello' and body 'How are you'", additional_kwargs={}, response_metadata={}, id='10b2fb0a-7231-42fa-81cd-7dfc2b4a1cb3'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 164, 'prompt_tokens': 179, 'total_tokens': 343, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQsP0kY9XJUlRpDOyTW167Y6Mqmfe', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c8bf-060d-76f3-b6ca-25bb9909041b-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'wrong

## Model call limit

Limit the number of model calls to prevent infinite loops or excessive costs. Model call limit is useful for the following:
- Preventing runaway agents from making too many API calls.
- Enforcing cost controls on production deployments.
- Testing agent behavior within specific call budgets.

In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-mini",
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=10,
            run_limit=5,
            exit_behavior="end",
        ),
    ],
)

In [28]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "test-call-limit"}}

# Same thread_id reused across calls -> thread_limit=10 counts cumulative model calls here
for i in range(12):
    result = agent.invoke(
        {"messages": [HumanMessage(content=f"Just say 'ok {i+1}'")]},
        config=config
    )
    last = result["messages"][-1]
    print(f"Call {i+1}: content={last.content!r} | jump_to={result.get('jump_to')}")


Call 1: content='ok 1' | jump_to=None
Call 2: content='ok 2' | jump_to=None
Call 3: content='ok 3' | jump_to=None
Call 4: content='ok 4' | jump_to=None
Call 5: content='ok 5' | jump_to=None
Call 6: content='ok 6' | jump_to=None
Call 7: content='ok 7' | jump_to=None
Call 8: content='ok 8' | jump_to=None
Call 9: content='ok 9' | jump_to=None
Call 10: content='ok 10' | jump_to=None
Call 11: content='Model call limits exceeded: thread limit (10/10)' | jump_to=None
Call 12: content='Model call limits exceeded: thread limit (10/10)' | jump_to=None
